In [1]:
import os
os.getcwd()

'C:\\Users\\PC'

In [2]:
os.chdir(r"D:\FQL\PJ 5")

## Section 1 : im[ort Libraries & Functions

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import joblib, os, warnings
warnings.filterwarnings('ignore')

In [6]:
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [7]:
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.dpi'] = 120
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

####  ────── HELPER FUNCTIONS ───────────────────────────

In [9]:
def encode_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    # Binary
    binary_map = {'Yes': 1, 'No': 0}
    for col in ['Family History', 'Smoking', 'Alcohol Consumption', 'Prior Fractures']:
        df[col] = df[col].map(binary_map)
    # Ordinal
    df['Hormonal Changes']  = df['Hormonal Changes'].map({'Normal': 0, 'Postmenopausal': 1})
    df['Body Weight']       = df['Body Weight'].map({'Underweight': 0, 'Normal': 1, 'Overweight': 2})
    df['Calcium Intake']    = df['Calcium Intake'].map({'Low': 0, 'Adequate': 1})
    df['Vitamin D Intake']  = df['Vitamin D Intake'].map({'Insufficient': 0, 'Sufficient': 1})
    df['Physical Activity'] = df['Physical Activity'].map({'Sedentary': 0, 'Moderate': 1, 'Active': 2})
    df['Gender']            = df['Gender'].map({'Female': 1, 'Male': 0})
    # One-hot
    df = pd.get_dummies(df, columns=['Race/Ethnicity', 'Medical Conditions', 'Medications'], drop_first=True)
    # Target
    if 'Osteoporosis' in df.columns:
        df['Osteoporosis'] = df['Osteoporosis'].map({'Yes': 1, 'No': 0})
    return df

In [11]:
def add_engineered_features(df: pd.DataFrame) -> pd.DataFrame:

    df = df.copy()
    df['Nutrient_Deficiency'] = ((df['Calcium Intake'] == 0) | (df['Vitamin D Intake'] == 0)).astype(int)
    df['Lifestyle_Risk']      = df['Smoking'] + df['Alcohol Consumption']
    df['Age_Group']           = pd.cut(df['Age'], bins=[0, 40, 55, 70, 120], labels=[0, 1, 2, 3]).astype(int)
    df['Hormonal_Bone_Risk']  = df['Hormonal Changes'] * df['Family History']
    return df

In [12]:
def plot_gridsearch_heatmap(grid_results: dict, param_x: str, param_y: str,
                            score_key: str = 'mean_test_score', title: str = '', ax=None):
    res_df = pd.DataFrame(grid_results)
    pivot = res_df.pivot_table(
        index=f'param_{param_y}',
        columns=f'param_{param_x}',
        values=score_key,
        aggfunc='mean')
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 5))
    sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlGn', ax=ax, linewidths=0.5)
    ax.set_title(title, fontsize=11, fontweight='bold')
    return ax

In [14]:
def summarize_metrics(trained_models: dict, X_test_dict: dict, y_test) -> pd.DataFrame:
    rows = []
    for name, info in trained_models.items():
        X_ts = X_test_dict[name]
        y_pred = info['model'].predict(X_ts)
        rows.append({
            'Model':      name,
            'CV F1':      round(info['cv_f1'], 4),
            'Test Acc':   round(accuracy_score(y_test, y_pred), 4),
            'Test F1':    round(f1_score(y_test, y_pred, zero_division=0), 4),
        })
    return pd.DataFrame(rows).sort_values('CV F1', ascending=False).reset_index(drop=True)
print('All helper functions defined.')

All helper functions defined.


## Section 2 : Load & Preprocessed Data

In [15]:
df_raw = pd.read_csv('osteoporosis.csv')
df_raw.drop(columns=['ID'], inplace=True, errors='ignore')

# Impute missing values BEFORE split to avoid introducing NaN during encoding
for col in df_raw.select_dtypes(include='number').columns:
    df_raw[col].fillna(df_raw[col].median(), inplace=True)
for col in df_raw.select_dtypes(include='object').columns:
    df_raw[col].fillna(df_raw[col].mode()[0], inplace=True)

print(f'Raw dataset: {df_raw.shape[0]} rows × {df_raw.shape[1]} cols')
print(f'Missing values: {df_raw.isnull().sum().sum()}')

Raw dataset: 1958 rows × 16 cols
Missing values: 0


## Section 3 : Data Leakage Audit 